In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
df_bookings = pd.read_csv(Path(r"C:\Users\krake\Downloads\Build1_DataPack\datapack\bookings\merged_bookings.csv"))
df_collections = pd.read_csv(Path(r"C:\Users\krake\Downloads\Build1_DataPack\datapack\collections\merged_collections.csv"))
df_results = pd.read_csv(Path(r"C:\Users\krake\Downloads\Build1_DataPack\datapack\results\merged_results.csv"))
df_bookings.columns

Index(['booking_id', 'created_at', 'slot_at', 'city', 'clinic_id', 'channel',
       'source', 'patient_id', 'patient_name', 'test_code', 'test_price',
       'home_fee', 'data_source'],
      dtype='str')

In [2]:
df_bookings_agg = (
    df_bookings.groupby('booking_id')
    .agg(
        created_at=('created_at', 'first'),
        slot_at=('slot_at', 'first'),
        city=('city', 'first'),
        clinic_id=('clinic_id', 'first'),
        channel=('channel', 'first'),
        source=('source', 'first'),
        patient_id=('patient_id', 'first'),
        patient_name=('patient_name', 'first'),
        n_tests=('test_code', 'count'),
        test_revenue=('test_price', 'sum'),
        home_fee=('home_fee', 'first'),
        data_source=('data_source', 'first')
    )
    .reset_index()
)
df_bookings_agg

,booking_id,created_at,slot_at,city,clinic_id,channel,source,patient_id,patient_name,n_tests,test_revenue,home_fee,data_source
0,BK-AHM-202501-000001,2025-01-07 21:46:00+00:00,2025-01-09 02:56:00+00:00,Ahmedabad,AHM-03,Clinic,Walkin,PAHM0028525,Saanvi Bose,3,2000.0,0.0,LEGACY
1,BK-AHM-202501-000002,2025-01-02 23:48:00+00:00,2025-01-06 07:52:00+00:00,Ahmedabad,AHM-04,Clinic,App,PAHM0031384,Nisha Iyer,1,650.0,0.0,LEGACY
2,BK-AHM-202501-000003,2025-01-27 05:16:00+00:00,2025-01-28 08:07:00+00:00,Ahmedabad,AHM-01,Home,App,PAHM0027001,Sneha Menon,4,1670.0,150.0,LEGACY
3,BK-AHM-202501-000004,2025-01-19 21:18:00+00:00,2025-01-24 01:45:00+00:00,Ahmedabad,AHM-03,Clinic,App,PAHM0007069,Arjun Bose,1,160.0,0.0,LEGACY
4,BK-AHM-202501-000005,2025-01-06 02:51:00+00:00,2025-01-11 04:26:00+00:00,Ahmedabad,AHM-03,Clinic,App,PAHM0002520,Ritika Nair,2,1900.0,0.0,LEGACY
...,...,...,...,...,...,...,...,...,...,...,...,...,...
565288,NOVA_JAI_0055749,2026-06-01 23:19:00+00:00,2026-06-08 06:24:00+00:00,Jaipur,JAI-02,Home,App,PJAI0028824,NaN,2,1000.0,150.0,NOVA
565289,NOVA_JAI_0055750,2026-06-15 17:58:00+00:00,2026-06-18 02:52:00+00:00,Jaipur,JAI-01,Home,Walkin,PJAI0028764,NaN,3,1800.0,150.0,NOVA
565290,NOVA_JAI_0055751,2026-06-19 00:06:00+00:00,2026-06-26 06:17:00+00:00,Jaipur,JAI-02,Clinic,Walkin,PJAI0028485,NaN,1,1499.0,0.0,NOVA
565291,NOVA_JAI_0055752,2026-06-06 20:59:00+00:00,2026-06-12 05:48:00+00:00,Jaipur,JAI-02,Home,Phone,PJAI0017296,NaN,3,1320.0,150.0,NOVA


In [3]:
df_results.columns

Index(['booking_id', 'test_code', 'result_ts', 'result_value', 'result_flag',
       'city'],
      dtype='str')

In [4]:
res_by_booking = df_results.groupby('booking_id').agg(n_results=('test_code', 'count'))
bookings = df_bookings_agg.merge(res_by_booking, on ='booking_id', how = 'left')
bookings['n_results'] = bookings['n_results'].fillna(0).astype(int)

In [5]:
bookings.head(3)

,booking_id,created_at,slot_at,city,clinic_id,channel,source,patient_id,patient_name,n_tests,test_revenue,home_fee,data_source,n_results
0,BK-AHM-202501-000001,2025-01-07 21:46:00+00:00,2025-01-09 02:56:00+00:00,Ahmedabad,AHM-03,Clinic,Walkin,PAHM0028525,Saanvi Bose,3,2000.0,0.0,LEGACY,3
1,BK-AHM-202501-000002,2025-01-02 23:48:00+00:00,2025-01-06 07:52:00+00:00,Ahmedabad,AHM-04,Clinic,App,PAHM0031384,Nisha Iyer,1,650.0,0.0,LEGACY,1
2,BK-AHM-202501-000003,2025-01-27 05:16:00+00:00,2025-01-28 08:07:00+00:00,Ahmedabad,AHM-01,Home,App,PAHM0027001,Sneha Menon,4,1670.0,150.0,LEGACY,3


In [6]:
dups = df_collections.duplicated(subset=['booking_id', 'status', 'collected_at'], keep='first')
coll = df_collections[~dups]
bookings = bookings.merge(coll[['booking_id', 'status', 'collected_at']],
                          on='booking_id', how='left', validate='one_to_one')
bookings['status'] = bookings['status'].fillna('cancelled_or_missing')

In [7]:
bookings.columns

Index(['booking_id', 'created_at', 'slot_at', 'city', 'clinic_id', 'channel',
       'source', 'patient_id', 'patient_name', 'n_tests', 'test_revenue',
       'home_fee', 'data_source', 'n_results', 'status', 'collected_at'],
      dtype='str')

In [8]:
# Filter Jan–Jun 2025 and Jan–Jun 2026

mask = ((bookings['slot_at'].between('2025-01-01', '2025-06-30 23:59:59'))|(bookings['slot_at'].between('2026-01-01', '2026-06-30 23:59:59')))


In [9]:
df = bookings.loc[mask].copy()

In [10]:
# Convert slot_at to datetime
df['slot_at'] = pd.to_datetime(
    df['slot_at'], errors='coerce'
)

In [11]:
# Create half-year labels
df['period'] = df['slot_at'].dt.year


In [12]:
df.groupby('period').agg(
    total_bookings = ('booking_id','count'),
    total_tests = ('n_tests','sum'),
    total_tests_results = ('n_results','sum')
)

,total_bookings,total_tests,total_tests_results
period,,,
2025,189445,435008,354137
2026,194664,456788,371742


In [13]:
# Calculate patients served
patients = df.groupby('period')['patient_id'].nunique()
patients

period
2025    94461
2026    98014
Name: patient_id, dtype: int64

In [14]:
# Calculate total visits
visits = df.groupby('period')['booking_id'].nunique()
visits

period
2025    189445
2026    194664
Name: booking_id, dtype: int64

In [15]:
# Calculate visits per patient
visits_per_patient = visits / patients
visits_per_patient

period
2025    2.005537
2026    1.986084
dtype: float64

In [16]:
# Calculate total tests
tests = df.groupby('period')['n_tests'].sum()
tests

period
2025    435008
2026    456788
Name: n_tests, dtype: int64

In [17]:
# Calculate tests per visit
tests_per_visit = (tests / visits)
tests_per_visit

period
2025    2.296223
2026    2.346546
dtype: float64

In [18]:
# Combine everything into one DataFrame
combined_df = pd.DataFrame({
    'patients_served': patients,
    'total_visits': visits,
    'visits_per_patient': round(visits_per_patient,2),
    'total_tests': tests,
    'tests_per_visit': round(tests_per_visit,2)
}).reset_index()

combined_df

,period,patients_served,total_visits,visits_per_patient,total_tests,tests_per_visit
0,2025,94461,189445,2.01,435008,2.30
1,2026,98014,194664,1.99,456788,2.35


In [19]:
melt_df = pd.melt(
    combined_df,
    id_vars='period',
    var_name='metric',
    value_name='value'
)

# Compare 2025 and 2026
growth_df = melt_df.pivot(
    index='metric',
    columns='period',
    values='value'
).reset_index()

# Calculate percentage growth
growth_df['growth'] = (
    (growth_df[2026] - growth_df[2025])
    / growth_df[2025] * 100
).round(2)

growth_df

period,metric,2025,2026,growth
0,patients_served,94461.00,98014.00,3.76
1,tests_per_visit,2.30,2.35,2.17
2,total_tests,435008.00,456788.00,5.01
3,total_visits,189445.00,194664.00,2.75
4,visits_per_patient,2.01,1.99,-1.00


In [20]:
# We can see the drop in visits_per_patient over 2025 1st half and 2026 1st half
# So, now we will check over cities and periods

city = df.groupby(['city', 'period']).agg(
    patients=('patient_id', 'nunique'),
    tests=('n_tests', 'sum'),
    total_tests_results = ('n_results','sum')
).reset_index()

city

,city,period,patients,tests,total_tests_results
0,Ahmedabad,2025,16689,76727,62519
1,Ahmedabad,2026,18300,84816,69371
2,Bengaluru,2025,21656,99732,80977
3,Bengaluru,2026,23214,107355,88263
4,Chennai,2025,17553,80022,64888
5,Chennai,2026,16676,80636,63989
6,Hyderabad,2025,15700,73058,59670
7,Hyderabad,2026,16967,78450,64363
8,Jaipur,2025,9698,45017,36726
9,Jaipur,2026,8524,40164,32121


In [21]:
city_pivot = city.pivot(
    index='city',
    columns='period',
    values=['patients', 'tests','total_tests_results']
)

city_pivot

patients         tests         total_tests_results       
period        2025   2026   2025    2026                2025   2026
city                                                               
Ahmedabad    16689  18300  76727   84816               62519  69371
Bengaluru    21656  23214  99732  107355               80977  88263
Chennai      17553  16676  80022   80636               64888  63989
Hyderabad    15700  16967  73058   78450               59670  64363
Jaipur        9698   8524  45017   40164               36726  32121
Pune         13165  14333  60452   65367               49357  53635

In [22]:
# calculate patient growth and visit growth by city

city_growth = pd.DataFrame({
    'Patients_growth': (
        (city_pivot[('patients', 2026)] -
         city_pivot[('patients', 2025)]) /
        city_pivot[('patients', 2025)] * 100
    ).round(2),

    'Tests_growth': (
        (city_pivot[('tests', 2026)] -
         city_pivot[('tests', 2025)]) /
        city_pivot[('tests', 2025)] * 100
    ).round(2),

    'Tests_results_growth': (
            (city_pivot[('total_tests_results', 2026)] -
             city_pivot[('total_tests_results', 2025)]) /
            city_pivot[('total_tests_results', 2025)] * 100
        ).round(2)
}).reset_index()

city_growth

,city,Patients_growth,Tests_growth,Tests_results_growth
0,Ahmedabad,9.65,10.54,10.96
1,Bengaluru,7.19,7.64,9.00
2,Chennai,-5.00,0.77,-1.39
3,Hyderabad,8.07,7.38,7.86
4,Jaipur,-12.11,-10.78,-12.54
5,Pune,8.87,8.13,8.67


In [23]:
df['month'] = df['slot_at'].dt.month_name()

In [24]:
# we can see the drop in Jaipur and Chennai on Patients growth % over period
# Now we are checking month by month for this 2 cities to find in which month 

city_mask = df['city'].isin(['Jaipur','Chennai'])
df_city_mask = df[city_mask]
df_city_mask

,booking_id,created_at,slot_at,city,clinic_id,channel,source,patient_id,patient_name,n_tests,test_revenue,home_fee,data_source,n_results,status,collected_at,period,month
231247,BK-CHE-202501-000001,2025-01-18 04:26:00+00:00,2025-01-23 07:14:00+00:00,Chennai,CHE-01,Home,App,PCHE0034759,Pooja Mehta,1,120.0,150.0,LEGACY,1,collected,2025-01-23 13:03:00,2025,January
231248,BK-CHE-202501-000002,2025-01-03 18:50:00+00:00,2025-01-09 03:22:00+00:00,Chennai,CHE-01,Clinic,App,PCHE0018901,Manish Reddy,2,500.0,0.0,LEGACY,2,collected,2025-01-09 09:57:00,2025,January
231249,BK-CHE-202501-000003,2025-01-24 00:18:00+00:00,2025-01-29 03:54:00+00:00,Chennai,CHE-01,Home,App,PCHE0032062,Saanvi Desai,1,650.0,150.0,LEGACY,1,collected,2025-01-29 09:17:00,2025,January
231250,BK-CHE-202501-000004,2025-01-18 11:47:00+00:00,2025-01-21 13:59:00+00:00,Chennai,CHE-04,Home,App,PCHE0031311,Farhan Reddy,1,650.0,150.0,LEGACY,0,no_show,NaN,2025,January
231251,BK-CHE-202501-000005,2025-01-02 00:37:00+00:00,2025-01-07 08:09:00+00:00,Chennai,CHE-03,Home,App,PCHE0001397,Saanvi Joshi,3,1550.0,150.0,LEGACY,0,cancelled_or_missing,NaN,2025,January
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
565288,NOVA_JAI_0055749,2026-06-01 23:19:00+00:00,2026-06-08 06:24:00+00:00,Jaipur,JAI-02,Home,App,PJAI0028824,NaN,2,1000.0,150.0,NOVA,0,cancelled_or_missing,NaN,2026,June
565289,NOVA_JAI_0055750,2026-06-15 17:58:00+00:00,2026-06-18 02:52:00+00:00,Jaipur,JAI-01,Home,Walkin,PJAI0028764,NaN,3,1800.0,150.0,NOVA,0,cancelled_or_missing,NaN,2026,June
565290,NOVA_JAI_0055751,2026-06-19 00:06:00+00:00,2026-06-26 06:17:00+00:00,Jaipur,JAI-02,Clinic,Walkin,PJAI0028485,NaN,1,1499.0,0.0,NOVA,0,cancelled_or_missing,NaN,2026,June
565291,NOVA_JAI_0055752,2026-06-06 20:59:00+00:00,2026-06-12 05:48:00+00:00,Jaipur,JAI-02,Home,Phone,PJAI0017296,NaN,3,1320.0,150.0,NOVA,0,cancelled_or_missing,NaN,2026,June


In [25]:
bookings = (
    df_city_mask[df_city_mask['period'].isin([2025, 2026])]
    .groupby(['city', 'month', 'period']).agg(
        count_bookings =('booking_id','nunique')).reset_index()
)
bookings

,city,month,period,count_bookings
0,Chennai,April,2025,5820
1,Chennai,April,2026,5535
2,Chennai,February,2025,5605
3,Chennai,February,2026,6180
4,Chennai,January,2025,5713
5,Chennai,January,2026,5764
6,Chennai,June,2025,5811
7,Chennai,June,2026,5479
8,Chennai,March,2025,6106
9,Chennai,March,2026,5662


In [26]:
bookings_pivot = bookings.pivot(
    index=['city', 'month'],
    columns='period',
    values='count_bookings'
).reset_index()

bookings_pivot

period,city,month,2025,2026
0,Chennai,April,5820,5535
1,Chennai,February,5605,6180
2,Chennai,January,5713,5764
3,Chennai,June,5811,5479
4,Chennai,March,6106,5662
5,Chennai,May,5825,5571
6,Jaipur,April,2940,2754
7,Jaipur,February,3489,2929
8,Jaipur,January,3327,2905
9,Jaipur,June,3031,2869


In [27]:
bookings_pivot['person_visit_growth'] = round(((bookings_pivot[2026] - 
                            bookings_pivot[2025])/bookings_pivot[2025]
                            *100),2)

bookings_pivot

period,city,month,2025,2026,person_visit_growth
0,Chennai,April,5820,5535,-4.90
1,Chennai,February,5605,6180,10.26
2,Chennai,January,5713,5764,0.89
3,Chennai,June,5811,5479,-5.71
4,Chennai,March,6106,5662,-7.27
5,Chennai,May,5825,5571,-4.36
6,Jaipur,April,2940,2754,-6.33
7,Jaipur,February,3489,2929,-16.05
8,Jaipur,January,3327,2905,-12.68
9,Jaipur,June,3031,2869,-5.34


In [28]:
bookings_pivot[bookings_pivot['person_visit_growth'].le(0)]

period,city,month,2025,2026,person_visit_growth
0,Chennai,April,5820,5535,-4.90
3,Chennai,June,5811,5479,-5.71
4,Chennai,March,6106,5662,-7.27
5,Chennai,May,5825,5571,-4.36
6,Jaipur,April,2940,2754,-6.33
7,Jaipur,February,3489,2929,-16.05
8,Jaipur,January,3327,2905,-12.68
9,Jaipur,June,3031,2869,-5.34
10,Jaipur,March,3604,2940,-18.42
11,Jaipur,May,3188,2771,-13.08


In [29]:
monthly_city = (
    df.groupby(['city', 'period', 'month'])
      .agg(
          patients_served=('patient_id', 'nunique'),
          bookings=('booking_id', 'nunique'),
          tests=('n_tests', 'sum')
      )
      .reset_index()
)

monthly_city['visits_per_patient'] = (
    monthly_city['bookings'] / monthly_city['patients_served']
).round(2)

# Compare 2025 and 2026 for the same city and month
patient_comparison = monthly_city.pivot(
    index=['city', 'month'],
    columns='period',
    values=['patients_served', 'bookings', 'tests', 'visits_per_patient']
)

for metric in ['patients_served', 'bookings', 'tests']:
    patient_comparison[(metric, 'change')] = (
        patient_comparison[(metric, 2026)]
        - patient_comparison[(metric, 2025)]
    )

    patient_comparison[(metric, 'growth_pct')] = round((
        patient_comparison[(metric, 'change')]
        / patient_comparison[(metric, 2025)].replace(0, np.nan)
        * 100
    ),2)

patient_comparison.loc['Jaipur']

patients_served         bookings           tests          \
period              2025    2026     2025    2026    2025    2026   
month                                                               
April             2481.0  2304.0   2940.0  2754.0  6881.0  6434.0   
February          2860.0  2404.0   3489.0  2929.0  8053.0  6794.0   
January           2771.0  2419.0   3327.0  2905.0  7612.0  6729.0   
June              2522.0  2373.0   3031.0  2869.0  6909.0  6798.0   
March             2942.0  2469.0   3604.0  2940.0  8278.0  6897.0   
May               2681.0  2343.0   3188.0  2771.0  7284.0  6512.0   

         visits_per_patient       patients_served            bookings  \
period                 2025  2026          change growth_pct   change   
month                                                                   
April                  1.19  1.20          -177.0      -7.13   -186.0   
February               1.22  1.22          -456.0     -15.94   -560.0   
January                1.20  1.20          -352.0     -12.70   -422.0   
June                   1.20  1.21          -149.0      -5.91   -162.0   
March                  1.23  1.19          -473.0     -16.08   -664.0   
May                    1.19  1.18          -338.0     -12.61   -417.0   

                      tests             
period   growth_pct  change growth_pct  
month                                   
April         -6.33  -447.0      -6.50  
February     -16.05 -1259.0     -15.63  
January      -12.68  -883.0     -11.60  
June          -5.34  -111.0      -1.61  
March        -18.42 -1381.0     -16.68  
May          -13.08  -772.0     -10.60

In [30]:
channel_monthly = (
    df[df['city'].isin(['Jaipur', 'Chennai'])]
    .groupby(['city', 'channel', 'month', 'period'])
    .agg(bookings=('booking_id', 'nunique'))
    .reset_index()
)

channel_comparison = channel_monthly.pivot(
    index=['city', 'channel', 'month'],
    columns='period',
    values='bookings'
).fillna(0)

channel_comparison['booking_change'] = (
    channel_comparison[2026] - channel_comparison[2025]
)

channel_comparison['growth_pct'] = round((
    (channel_comparison[2026] - channel_comparison[2025])
    / channel_comparison[2025].replace(0, np.nan) * 100
),2)

# Show the largest absolute booking losses first
channel_comparison.sort_values('booking_change').reset_index()

period,city,channel,month,2025,2026,booking_change,growth_pct
0,Chennai,Clinic,March,3985,2879,-1106,-27.75
1,Jaipur,Clinic,March,2357,1471,-886,-37.59
2,Chennai,Clinic,January,3768,2943,-825,-21.89
3,Jaipur,Clinic,February,2332,1549,-783,-33.58
4,Jaipur,Clinic,January,2171,1476,-695,-32.01
5,Chennai,Clinic,June,3906,3301,-605,-15.49
6,Chennai,Clinic,April,3857,3290,-567,-14.70
7,Chennai,Clinic,May,3898,3344,-554,-14.21
8,Chennai,Clinic,February,3677,3176,-501,-13.63
9,Jaipur,Clinic,May,2101,1649,-452,-21.51


In [31]:
channel_loss = (
    df[df['city'].isin(['Jaipur', 'Chennai'])]
    .groupby(['city', 'channel', 'period'])
    .agg(bookings=('booking_id', 'nunique'))
    .reset_index()
    .pivot(index=['city', 'channel'], columns='period', values='bookings')
    .fillna(0)
)

channel_loss['booking_change'] = channel_loss[2026] - channel_loss[2025]
channel_loss['absolute_loss'] = (-channel_loss['booking_change']).clip(lower=0)
channel_loss['growth_pct'] = (
    channel_loss['booking_change']
    / channel_loss[2025].replace(0, np.nan)
    * 100
).round(2)

channel_loss.sort_values('booking_change').reset_index()

period,city,channel,2025,2026,booking_change,absolute_loss,growth_pct
0,Chennai,Clinic,23091,18933,-4158,4158,-18.01
1,Jaipur,Clinic,12889,9541,-3348,3348,-25.98
2,Jaipur,Home,6690,7627,937,0,14.01
3,Chennai,Home,11789,15258,3469,0,29.43


In [32]:
clinic_monthly = (
    df.groupby(['city', 'clinic_id', 'month', 'period'])
      .agg(bookings=('booking_id', 'nunique'))
      .reset_index()
)

clinic_comparison = clinic_monthly.pivot(
    index=['city', 'clinic_id', 'month'],
    columns='period',
    values='bookings'
)

clinic_comparison = clinic_comparison.reindex(
    columns=[2025, 2026]
).fillna(0)

clinic_comparison['booking_change'] = (
    clinic_comparison[2026] - clinic_comparison[2025]
)

clinic_comparison['clinic_type'] = np.where(
    clinic_comparison[2025] == 0,
    'New / no 2025 bookings',
    'Existing'
)

clinic_comparison['growth_pct'] = (
    clinic_comparison['booking_change']
    / clinic_comparison[2025].replace(0, np.nan)
    * 100
)

clinic_comparison.sort_values('booking_change').reset_index()

period,city,clinic_id,month,2025,2026,booking_change,clinic_type,growth_pct
0,Jaipur,JAI-01,March,1364.0,1114.0,-250.0,Existing,-18.328446
1,Jaipur,JAI-01,February,1350.0,1123.0,-227.0,Existing,-16.814815
2,Jaipur,JAI-02,March,954.0,728.0,-226.0,Existing,-23.689727
3,Jaipur,JAI-01,May,1275.0,1062.0,-213.0,Existing,-16.705882
4,Chennai,CHE-02,March,1447.0,1258.0,-189.0,Existing,-13.061507
...,...,...,...,...,...,...,...,...
180,Ahmedabad,AHM-01,March,1899.0,2123.0,224.0,Existing,11.795682
181,Ahmedabad,AHM-01,May,1858.0,2087.0,229.0,Existing,12.325081
182,Ahmedabad,AHM-02,February,1202.0,1446.0,244.0,Existing,20.299501
183,Ahmedabad,AHM-01,February,1808.0,2122.0,314.0,Existing,17.367257


In [33]:
clinic_summary = (
    clinic_comparison.reset_index()
    .groupby(['city', 'month'])
    .agg(
        clinics_compared=('clinic_id', 'nunique'),
        clinics_declining=('booking_change', lambda s: (s < 0).sum()),
        clinics_growing=('booking_change', lambda s: (s > 0).sum()),
        total_booking_change=('booking_change', 'sum')
    )
    .reset_index()
)

clinic_summary

,city,month,clinics_compared,clinics_declining,clinics_growing,total_booking_change
0,Ahmedabad,April,5,1,4,194.0
1,Ahmedabad,February,5,0,5,805.0
2,Ahmedabad,January,5,1,4,348.0
3,Ahmedabad,June,5,0,5,385.0
4,Ahmedabad,March,5,0,5,665.0
5,Ahmedabad,May,5,0,5,454.0
6,Bengaluru,April,6,0,6,520.0
7,Bengaluru,February,6,0,6,1110.0
8,Bengaluru,January,6,1,5,274.0
9,Bengaluru,June,6,1,5,73.0


In [34]:
# Full H1 booking-source analysis for Jaipur and Chennai (includes January–June)
source_summary = (
    df[df['city'].isin(['Jaipur', 'Chennai']) & df['channel'].eq('Clinic')]
    .groupby(['city', 'source', 'period'])
    .agg(bookings=('booking_id', 'nunique'))
    .reset_index()
)

source_comparison = source_summary.pivot(
    index=['city', 'source'],
    columns='period',
    values='bookings'
).reindex(columns=[2025, 2026])

source_comparison['booking_change'] = source_comparison[2026] - source_comparison[2025]
source_comparison['growth_pct'] = (
    source_comparison['booking_change']
    / source_comparison[2025].replace(0, np.nan)
    * 100
).round(2)

source_comparison.reset_index().sort_values(['city', 'source'])

period,city,source,2025,2026,booking_change,growth_pct
0,Chennai,App,12016,9774,-2242,-18.66
1,Chennai,Phone,7425,6168,-1257,-16.93
2,Chennai,Walkin,3650,2991,-659,-18.05
3,Jaipur,App,7089,5204,-1885,-26.59
4,Jaipur,Phone,3823,2922,-901,-23.57
5,Jaipur,Walkin,1977,1415,-562,-28.43


In [35]:
# One row per booking, with the number of tests ordered
booking_level = (
    df.groupby('booking_id')
      .agg(
          city=('city', 'first'),
          period=('period', 'first'),
          month=('month', 'first'),
          tests_ordered=('n_tests', 'sum')
      )
      .reset_index()
)

# One row per booking, with the number of recorded result rows
result_level = (
    df_results.groupby('booking_id')
         .agg(tests_with_results=('test_code', 'count'))
         .reset_index()
)

# Join at booking level to avoid multiplying booking rows
reconciliation = booking_level.merge(
    result_level,
    on='booking_id',
    how='left',
    validate='one_to_one'
)

reconciliation['tests_with_results'] = (
    reconciliation['tests_with_results'].fillna(0)
)

reconciliation['result_gap'] = (
    reconciliation['tests_ordered']
    - reconciliation['tests_with_results']
)

reconciliation['has_result_gap'] = reconciliation['result_gap'] > 0

In [36]:
results_summary = (
    reconciliation.groupby(['city', 'period'])
    .agg(
        bookings=('booking_id', 'nunique'),
        tests_ordered=('tests_ordered', 'sum'),
        tests_with_results=('tests_with_results', 'sum'),
        bookings_with_gap=('has_result_gap', 'sum'),
        total_result_gap=('result_gap', 'sum')
    )
    .reset_index()
)

results_summary['result_coverage_pct'] = (
    results_summary['tests_with_results']
    / results_summary['tests_ordered'].replace(0, np.nan)
    * 100
)

results_summary

,city,period,bookings,tests_ordered,tests_with_results,bookings_with_gap,total_result_gap,result_coverage_pct
0,Ahmedabad,2025,33369,76727,62519.0,6849,14208.0,81.482399
1,Ahmedabad,2026,36220,84816,69371.0,7293,15445.0,81.789992
2,Bengaluru,2025,43618,99732,80977.0,8961,18755.0,81.194602
3,Bengaluru,2026,45678,107355,88263.0,9060,19092.0,82.216012
4,Chennai,2025,34880,80022,64888.0,7296,15134.0,81.087701
5,Chennai,2026,34191,80636,63989.0,7710,16647.0,79.355375
6,Hyderabad,2025,31712,73058,59670.0,6481,13388.0,81.674834
7,Hyderabad,2026,33436,78450,64363.0,6670,14087.0,82.043340
8,Jaipur,2025,19579,45017,36726.0,4036,8291.0,81.582513
9,Jaipur,2026,17168,40164,32121.0,3757,8043.0,79.974604


In [37]:
results_monthly = (
    reconciliation.groupby(['city', 'month', 'period'])
    .agg(
        tests_ordered=('tests_ordered', 'sum'),
        tests_with_results=('tests_with_results', 'sum'),
        result_gap=('result_gap', 'sum')
    )
    .reset_index()
)

results_monthly

,city,month,period,tests_ordered,tests_with_results,result_gap
0,Ahmedabad,April,2025,13188,10794.0,2394.0
1,Ahmedabad,April,2026,13902,11241.0,2661.0
2,Ahmedabad,February,2025,12516,10095.0,2421.0
3,Ahmedabad,February,2026,14507,11895.0,2612.0
4,Ahmedabad,January,2025,12786,10464.0,2322.0
...,...,...,...,...,...,...
67,Pune,June,2026,10789,8747.0,2042.0
68,Pune,March,2025,9827,7976.0,1851.0
69,Pune,March,2026,11416,9461.0,1955.0
70,Pune,May,2025,9861,8092.0,1769.0


## Final interpretation: hypotheses and conclusions

**Scope:** H1 2025 (January–June) compared with H1 2026. These are descriptive comparisons, not formal statistical tests; they identify patterns but do not prove business causality.

| Hypothesis | Metric checked | Finding | Status |
|---|---|---|---|
| 1. The growth shortfall is related to modest patient growth | Unique patients served | 94,461 → 98,014 (**+3.76%**) while tests grew **+5.01%** | **Partially supported**; patient growth is modest, but this alone does not explain the target gap |
| 2. Existing patients visit less frequently | Visits per patient | 2.01 → 1.99 (approximately **−1.0%**) | **Weak directional support**; small decline, not proof of cause |
| 3. Test volume is weak because fewer tests are ordered per visit | Tests per visit | 2.30 → 2.35 (approximately **+2.2%**) | **Not supported**; tests per visit increased |
| 4. Weak growth is concentrated in particular cities | City-level test and patient growth | Jaipur tests **−10.78%**, patients **−12.11%**; Chennai tests **+0.77%**, patients **−5.00%** | **Supported descriptively**; Jaipur is the clearest concern and Chennai is nearly flat |
| 5. Clinic-channel weakness is contributing in Jaipur and Chennai | Full six-month bookings by channel | Chennai Clinic: 23,091 → 18,933 (**−18.0%**); Jaipur Clinic: 12,889 → 9,541 (**−26.0%**). Home bookings grew in both cities. | **Supported as a pattern**, not yet proven as the underlying cause |
| 6. One clinic alone explains the clinic-channel decline | Monthly booking changes by clinic | Declines are spread across multiple existing clinics in Jaipur and Chennai; new/no-baseline clinics must be treated separately | **Single-clinic explanation not supported by current descriptive evidence** |
| 7. One booking source alone explains the clinic-channel decline | Full six-month source comparison (App, Phone, Walk-in) | Use the corrected source comparison table above; compare the changes across all three sources | **Do not claim one source is the cause without a clearly isolated pattern** |
| 8. Booking test counts and recorded results align | Tests ordered vs. result rows | Overall counts differ substantially: 435,008 ordered vs. 354,137 result rows in 2025; 456,788 vs. 371,742 in 2026 | **Not reconciled**; check result coverage, data definitions, and joins before using result counts as interchangeable with ordered tests |

### Overall conclusion

Test volume grew by **5.01%**, from **435,008 to 456,788**, against an **18% target**—a shortfall of approximately **13 percentage points**. The strongest descriptive leads are Jaipur's decline and weak growth in Chennai. Overall patient growth was modest and visits per patient declined slightly, while tests per visit increased. Full six-month channel analysis shows clinic bookings declined materially in Jaipur and Chennai while home bookings increased, so the clinic channel deserves investigation. These results identify where to investigate next; they do **not** establish statistical significance or prove the operational root cause.

### Before presenting
1. Treat the booking-level test count as the primary test-volume metric unless its definition is confirmed.
2. Reconcile test orders with result rows and inspect the booking/result join and result timing.
3. Investigate Jaipur first, then Chennai, focusing on patient acquisition/retention and clinic-channel performance.
4. Do not calculate growth for a clinic with zero or missing 2025 baseline; label it as new/no baseline instead.
